# Tørrfall gaps — find unusually large polygons (possible brakkvannspoll)

Analyses `geonorge/torrfall_original_vs_filled_25833.gpkg`, produced by
`mnk.sources.prepare_torrfall_filled`. The `torrfall_gaps_added` layer holds the
standalone water pieces that `mnk.vectorize.fill_torrfall_gaps` found enclosed
between tørrfall and land and merged into tørrfall.

Most gaps are tiny coastline slivers, but a few are **large** — likely real
enclosed water bodies (brackish-water pools / *brakkvannspoll*). We look at the
size distribution, derive an **outlier threshold** statistically (Tukey fence in
log-area space, robust to the extreme right-skew), and save the large outliers
to `possible_brakkvannspoll_25833.geo.parquet`.

Run from the repo root (same `cwd` convention as notebooks 03/04).

## 0. Setup

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyogrio import read_dataframe, read_info

CRS = "EPSG:25833"
REPO = Path.cwd()

GPKG = REPO / "geonorge/torrfall_original_vs_filled_25833.gpkg"
GAPS_LAYER = "torrfall_gaps_added"
OUT_PATH = REPO / "geonorge/possible_brakkvannspoll_25833.geo.parquet"

# None -> use the statistical outlier threshold derived in section 2.
# Set a number (hectares) to override, e.g. MIN_AREA_HA_OVERRIDE = 5.0
MIN_AREA_HA_OVERRIDE = None

## 1. Layer overview (original vs filled)

How much the gap-fill added on top of the original tørrfall.

In [ ]:
for lyr in ["torrfall_original", GAPS_LAYER, "torrfall_filled"]:
    print(f"{lyr:22s} {read_info(GPKG, layer=lyr)['features']:>12,} features")

## 2. Size distribution and outlier threshold

Pull only `ST_Area(geom)` for every gap (no geometry) so the 1.8 M-row layer
stays cheap. The distribution spans orders of magnitude and is dominated by
sub-m² slivers, so the Tukey fence (`Q3 + 1.5·IQR`) is computed on **log10 area**
— the natural scale for multiplicative, heavy-tailed data.

In [ ]:
areas = read_dataframe(
    GPKG,
    sql=f"SELECT ST_Area(geom) AS area_m2 FROM {GAPS_LAYER}",
    sql_dialect="SQLITE",
    read_geometry=False,
)["area_m2"].to_numpy()


def log_tukey_upper_fence(x, k=1.5):
    """Upper Tukey fence (Q3 + k·IQR) computed in log10 space, back in m²."""
    lx = np.log10(x[x > 0])
    q1, q3 = np.percentile(lx, [25, 75])
    return 10.0 ** (q3 + k * (q3 - q1))


OUTLIER_AREA_M2 = log_tukey_upper_fence(areas)

print(f"gaps: {len(areas):,}")
print(pd.Series(areas).describe(percentiles=[0.5, 0.9, 0.99, 0.999]).round(1))
print(f"\noutlier fence (log Tukey 1.5·IQR): "
      f"{OUTLIER_AREA_M2:,.0f} m²  ({OUTLIER_AREA_M2 / 1e4:.1f} ha)  "
      f"-> {(areas >= OUTLIER_AREA_M2).sum():,} outliers")
print("\ncount of gaps at/above area thresholds:")
for ha in [0.1, 0.5, 1, 5, 10, 50, 100]:
    print(f"  >= {ha:>5} ha: {(areas >= ha * 1e4).sum():>8,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

pos = areas[areas > 0]
axes[0].hist(np.log10(pos), bins=80, color="#4C78A8")
axes[0].axvline(np.log10(OUTLIER_AREA_M2), color="crimson", ls="--",
                label=f"outlier fence = {OUTLIER_AREA_M2 / 1e4:.1f} ha")
axes[0].set(xlabel="log10(area [m²])", ylabel="gap count",
            title="All gap areas (log scale)")
axes[0].legend()

big = areas[areas >= OUTLIER_AREA_M2] / 1e4
axes[1].hist(big, bins=30, color="#E45756")
axes[1].set(xlabel="area [ha]", ylabel="count",
            title=f"Outliers only (n={len(big):,})")

plt.tight_layout()
plt.show()

## 3. Select the large outliers

Threshold = the outlier fence from §2 (or `MIN_AREA_HA_OVERRIDE` if set). Load
geometry **only** for gaps above it, then add shape metrics. `polsby_popper`
(4π·area / perimeter²; 1 = circle, →0 = thin) is informational — real basins are
often elongated, so it is not used as a filter.

In [ ]:
MIN_AREA_M2 = MIN_AREA_HA_OVERRIDE * 1e4 if MIN_AREA_HA_OVERRIDE else OUTLIER_AREA_M2
print(f"threshold: {MIN_AREA_M2:,.0f} m²  ({MIN_AREA_M2 / 1e4:.2f} ha)")

cand = read_dataframe(
    GPKG,
    sql=f"SELECT ST_Area(geom) AS area_m2, geom FROM {GAPS_LAYER} "
        f"WHERE ST_Area(geom) >= {MIN_AREA_M2}",
    sql_dialect="SQLITE",
)
if cand.crs is None:
    cand = cand.set_crs(CRS)

cand["area_ha"] = cand["area_m2"] / 1e4
cand["perim_m"] = cand.geometry.length
cand["polsby_popper"] = 4 * np.pi * cand["area_m2"] / cand["perim_m"] ** 2
cand = cand[["area_m2", "area_ha", "perim_m", "polsby_popper", "geometry"]]
cand = cand.sort_values("area_m2", ascending=False).reset_index(drop=True)

print(f"candidates: {len(cand):,}  (total {cand['area_ha'].sum():,.0f} ha)")
cand.drop(columns="geometry").head(20)

In [ ]:
ax = cand.plot(figsize=(7, 9), color="#4C78A8", edgecolor="k", linewidth=0.3)
ax.set_title(f"{len(cand):,} large gap polygons (possible brakkvannspoll)")
ax.set_axis_off()
plt.show()

## 4. Save candidates

In [ ]:
cand.to_parquet(OUT_PATH, compression="snappy")
print(f"Saved {len(cand):,} candidate polygons -> {OUT_PATH}")